In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cu128


In [ ]:
#Runs tensor through each layer, making the data more specific over time until it's between 10 images
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

print(model)

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=10, bias=True)
)


In [ ]:
image = torch.rand(1, 784)
#Uses model created in previous cell
output = model(image)

print("Output shape: ", output.shape)
print("Output: ", output)

Output shape:  torch.Size([1, 10])
Output:  tensor([[-0.1301,  0.0684, -0.0877, -0.1163, -0.0672, -0.0621,  0.0519, -0.1023,
          0.0118, -0.0559]], grad_fn=<AddmmBackward0>)


In [ ]:
from torchvision import datasets, transforms

#Transforms images into Tensors
transform = transforms.ToTensor()

#Imports set of 60,000 pictures of numbers meant for training
train_data = datasets.MNIST(root = 'data', train = True,
                            download = True, transform = transform)
#Imports set of 10,000 pictures meant for testing
test_data = datasets.MNIST(root = 'data', train = False,
                           download = True, transform = transform)

print('Traning Images: ', len(train_data))
print('Test Images: ', len(test_data))

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 476kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.43MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.32MB/s]

Traning Images:  60000
Test Images:  10000


In [ ]:
#DataLoader feed groups of images in batches instead of all at once
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

print("Number of batches in traning: ", len(train_loader))
print("Number of batches in testing: ", len(test_loader))

Number of batches in traning:  1875
Number of batches in testing:  313


In [ ]:
import torch.optim as optim

#Measures how wrong the model is after its prediction - Higher return number from the function means more incorrect
criterion = nn.CrossEntropyLoss()

#Changes the models direction after the prediction - parameters give access to all weights and lr is amount of change
#Less accurate weights are adjusted more
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#Number of times the model goes through each image
epochs = 5

#Completes loop 3 times, resets running loss after every epoch, starts epoch at 0
for epoch in range(epochs):
  running_loss = 0.0

#grabs the images 32 at a time along with their labels(what number they actually are)
  for images, labels in train_loader:
    #Reshapes each image from a 28x28 grid into a row of 784 numbers; -1 gives python permission to figure out new single dimension automatically
    images = images.view(images.shape[0], -1)

    #Sets optimizer gradient to zero for each new batch
    optimizer.zero_grad()
    #Runs batch of 32 images through the model
    output = model(images)
    #Calculates one loss number for the whole batch after comparing the model's predictions witht the image labels
    loss = criterion(output, labels)
    #PyTorch works backwards and determines how much each weight contributed to the error
    loss.backward()
    #Runs optimizer to nudge weights
    optimizer.step()

    #Adds up total batch loss and converts it from a tensor into a number through item()
    running_loss += loss.item()

  #presents loss as total loss divided by number of batches
  print(f"Epoch {epoch+1} loss: {running_loss/len(train_loader):.4f}")



Epoch 1 loss: 0.2872
Epoch 2 loss: 0.1190
Epoch 3 loss: 0.0831
Epoch 4 loss: 0.0615
Epoch 5 loss: 0.0476


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.shape[0], -1)
        output = model(images)
        predicted = torch.argmax(output, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 97.64%
